# X-MACE ASE Calculator: Predicting Properties from a Trained Model

This notebook shows how to load a trained X-MACE model and use it as an **ASE Calculator** to predict properties — energies, forces, NACs, SOCs — for any molecular geometry.

The calculator (`mace.py`) wraps the trained PyTorch model in the standard ASE `Calculator` interface, so it works with any ASE workflow out of the box.

### What we cover
1. Imports and setup
2. Loading a structure from an XYZ file
3. Creating the `MACECalculator`
4. Predicting energies and forces
5. Predicting NACs (non-adiabatic couplings)
6. Predicting SOCs (spin-orbit couplings)

---
> **Prerequisite:** A trained model file, e.g. `energies_forces.model`, produced by `scripts/run_train.py`  
> **Calculator file:** `mace.py` (the custom X-MACE ASE calculator)

## 1. Imports and Setup

In [5]:
import numpy as np
import torch
import ase.io
from ase import Atoms

# Import the custom X-MACE calculator
# Make sure mace.py is in the same directory (or add its path to sys.path)
import sys
sys.path.insert(0, ".")   # adjust if mace.py lives elsewhere
from mace.calculators import MACECalculator

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


## 2. Loading a Structure

X-MACE works with standard ASE `Atoms` objects. You can load structures from any format ASE supports: XYZ, VASP POSCAR, CIF, etc.

In [6]:
# --- Option A: Load a single frame from an XYZ file ---
atoms = ase.io.read("SINGLET_SOC_ALL.xyz", index=0)

# --- Option B: Load all frames ---
# all_frames = ase.io.read("SINGLET_SOC_ALL.xyz", index=":")

# --- Option C: Build a structure from scratch ---
# from ase.build import molecule
# atoms = molecule("H2O")

print(f"Number of atoms  : {len(atoms)}")
print(f"Chemical formula : {atoms.get_chemical_formula()}")
print(f"Positions shape  : {atoms.positions.shape}")
print(f"Available info keys: {list(atoms.info.keys())}")

Number of atoms  : 23
Chemical formula : H23
Positions shape  : (23, 3)
Available info keys: ['REF_energy', 'REF_forces', 'REF_smooth_nacs', 'REF_socs']


## 3. Creating the MACECalculator

The `MACECalculator` wraps the trained PyTorch model. The most important arguments are:

| Argument | Description | Example |
|---|---|---|
| `model_paths` | Path to the `.model` file (wildcard `*` for committees) | `"energies_forces.model"` |
| `device` | `"cuda"` or `"cpu"` | `"cuda"` |
| `n_energies` | Number of electronic states the model was trained on | `4` |
| `default_dtype` | Must match training dtype | `"float32"` |
| `model_type` | `"MACE"`, `"DipoleMACE"`, or `"EnergyDipoleMACE"` | `"MACE"` |

In [8]:
# Path to the model produced by training
# Checkpoints are saved in ./checkpoints/ and final model as <name>.model
MODEL_PATH = "checkpoints/energies_forces_run-100.model"   # <-- update this path

calc = MACECalculator(
    model_paths=MODEL_PATH,
    device=device,
    default_dtype="float32",   # match training dtype
    model_type="MACE",
)

print(f"Calculator loaded successfully.")

RuntimeError: CUDA error: unspecified launch failure
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## 4. Predicting Energies and Forces

Attach the calculator to an `Atoms` object and call the standard ASE property getters. Because X-MACE models **multiple electronic states**, `energy` returns an array of shape `(n_states,)` rather than a scalar.

In [4]:
# Attach the calculator
atoms.calc = calc

# Trigger calculation — ASE calls calc.calculate() internally
predicted_energies = atoms.get_potential_energy()   # returns array (n_states,)
predicted_forces   = calc.results["forces"]          # shape (N_atoms, n_states, 3)

print("=== Energies ===")
print(f"Shape  : {np.array(predicted_energies).shape}")
for i, e in enumerate(np.array(predicted_energies).flatten()):
    print(f"  State S{i}: {e:.6f} eV")

print()
print("=== Forces ===")
print(f"Shape  : {predicted_forces.shape}   (N_atoms, n_states, 3)")
print(f"Max |F| across all states: {np.abs(predicted_forces).max():.4f} eV/Å")
for s in range(4):
    f_rms = np.sqrt(np.mean(predicted_forces[:, s, :] ** 2))
    print(f"  State S{s} RMS force: {f_rms:.4f} eV/Å")

=== Energies ===
Shape  : (1, 4)
  State S0: -23090.769531 eV
  State S1: -23090.226562 eV
  State S2: -23090.773438 eV
  State S3: -23090.666016 eV

=== Forces ===
Shape  : (23, 4, 3)   (N_atoms, n_states, 3)
Max |F| across all states: 1.1388 eV/Å
  State S0 RMS force: 0.3360 eV/Å
  State S1 RMS force: 0.1347 eV/Å
  State S2 RMS force: 0.2731 eV/Å
  State S3 RMS force: 0.3132 eV/Å


### Understanding the output shapes

| `calc.results` key | Shape | Description |
|---|---|---|
| `energy` | `(n_states,)` | Per-state potential energies |
| `free_energy` | `(n_states,)` | Same as energy (alias) |
| `forces` | `(N_atoms, n_states, 3)` | Per-atom, per-state force vectors |
| `node_energy` | `(N_atoms, n_states)` | Atomic decomposition of energy |
| `stress` | `(6,)` Voigt | Stress tensor (periodic systems only) |

## 5. Predicting NACs (Non-Adiabatic Couplings)

If you trained a model with `--compute_nacs`, the `out` dictionary from the model will contain NAC predictions. The calculator stores them in `calc.results` after calling `calculate()`. 

To expose NACs through the calculator, enable them in the model's `calculate()` call by uncommenting the relevant lines in `mace.py` (the lines with `ret_tensors["nacs"]` and `self.results["smooth_nacs"]`). Once enabled:

In [ ]:
# After enabling NACs in mace.py:
#
#   In calculate():
#       ret_tensors["nacs"] = out["nacs"].detach()
#   In results:
#       self.results["smooth_nacs"] = ret_tensors["nacs"].cpu().numpy()
#
# Then retrieve them here:

atoms.calc = calc
atoms.get_potential_energy()   # trigger calculation

if "smooth_nacs" in calc.results:
    nacs = calc.results["smooth_nacs"]   # shape (N_atoms, n_pairs, 3)
    n_pairs = N_STATES * (N_STATES - 1) // 2
    print(f"NAC shape  : {nacs.shape}   (N_atoms, n_pairs, 3)")
    print(f"n_pairs    : {n_pairs}  [= {N_STATES}×({N_STATES}-1)/2]")
    print(f"Max |NAC|  : {np.abs(nacs).max():.4f}")
else:
    print("NACs not found in results.")
    print("Uncomment the NAC lines in mace.py's calculate() method and retrain with --compute_nacs.")

## 6. Predicting SOCs (Spin-Orbit Couplings)

Similarly, SOC predictions are available when the model was trained with `--compute_socs`. Enable them by uncommenting the `socs` lines in `mace.py`.

In [ ]:
# After enabling SOCs in mace.py:
#
#   In calculate():
#       ret_tensors["socs"] = out["socs"].detach()
#   In results:
#       self.results["socs"] = ret_tensors["socs"].cpu().numpy()

if "socs" in calc.results:
    socs = calc.results["socs"]   # shape (1, soc_num) e.g. (1, 252)
    print(f"SOC shape  : {socs.shape}")
    print(f"SOC values (first 10): {socs.flatten()[:10]}")
else:
    print("SOCs not found in results.")
    print("Uncomment the SOC lines in mace.py's calculate() method and train with --compute_socs.")

## 7. Running Predictions Over a Dataset

A common workflow: run the trained model over all frames in the dataset and collect predictions alongside reference values for error analysis.

In [ ]:
# Load the full dataset (or a subset for speed)
frames = ase.io.read("SINGLET_SOC_ALL.xyz", index=":100")   # first 100 frames

pred_energies = []   # predicted,  shape (n_frames, n_states)
ref_energies  = []   # reference,  shape (n_frames, n_states)
pred_forces   = []   # predicted,  shape (n_frames, N_atoms, n_states, 3)
ref_forces    = []   # reference,  shape (n_frames, N_atoms, n_states, 3)

print(f"Running predictions on {len(frames)} frames ...")
for i, frame in enumerate(frames):
    frame.calc = calc
    frame.get_potential_energy()   # triggers calculate()

    pred_energies.append(np.array(calc.results["energy"]).flatten())
    pred_forces.append(calc.results["forces"])                         # (N_atoms, n_states, 3)

    # Reference values stored in atoms.info by ASE when reading the XYZ
    ref_e = np.array(frame.info.get("energy", frame.info.get("REF_energy", None)))
    ref_f = np.array(frame.info.get("forces", frame.info.get("REF_forces", None)))
    if ref_e is not None:
        ref_energies.append(ref_e.flatten())
    if ref_f is not None:
        ref_forces.append(ref_f)

    if (i + 1) % 20 == 0:
        print(f"  Processed {i + 1}/{len(frames)} frames")

pred_energies = np.array(pred_energies)   # (n_frames, n_states)
pred_forces   = np.array(pred_forces)     # (n_frames, N_atoms, n_states, 3)
if ref_energies:
    ref_energies = np.array(ref_energies)
if ref_forces:
    ref_forces = np.array(ref_forces)

print(f"\nPredicted energies shape : {pred_energies.shape}")
print(f"Predicted forces shape   : {pred_forces.shape}")

### Compute MAE for energies and forces

In [ ]:
if len(ref_energies) > 0 and len(ref_forces) > 0:
    energy_mae = np.abs(pred_energies - ref_energies).mean(axis=0)   # per state
    forces_mae = np.abs(pred_forces   - ref_forces  ).mean()          # scalar

    print("Energy MAE per state (eV):")
    for s, mae in enumerate(energy_mae):
        print(f"  S{s}: {mae * 1000:.2f} meV")

    print(f"\nForce MAE (all states, all atoms): {forces_mae * 1000:.2f} meV/Å")
else:
    print("Reference data not found in dataset info keys — skipping MAE calculation.")